# Descriptive Statistics — News Analyst Ratings
Exploratory analysis covering:
1. Headline character length distribution
2. Articles per publisher
3. Publication volume trends over time

In [1]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.eda import (
    load_data,
    headline_length_stats,
    plot_headline_length_distribution,
    articles_per_publisher,
    plot_articles_per_publisher,
    publication_date_trends,
    plot_publication_trends,
    top_publishing_days,
)

DATA_PATH = 'data/raw/raw_analyst_ratings.csv'
OUTPUT_DIR = Path('data/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = load_data(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
df.head()

## 1. Headline Character Length

In [ ]:
stats = headline_length_stats(df)
print('Headline length statistics:')
print(stats.to_string())

In [ ]:
fig = plot_headline_length_distribution(df, save_path=str(OUTPUT_DIR / 'headline_length_dist.png'))
plt.show()

## 2. Articles per Publisher

In [ ]:
publisher_counts = articles_per_publisher(df, top_n=20)
print(f'Total unique publishers: {df["publisher"].nunique()}')
print('\nTop 20 publishers:')
print(publisher_counts.to_string())

In [ ]:
fig = plot_articles_per_publisher(df, top_n=20, save_path=str(OUTPUT_DIR / 'articles_per_publisher.png'))
plt.show()

## 3. Publication Volume Trends Over Time

In [ ]:
weekly = publication_date_trends(df, freq='W')
print(f'Weekly trend — {len(weekly)} weeks')
print(f'Peak week: {weekly.idxmax().date()} ({weekly.max()} articles)')
print(f'Average per week: {weekly.mean():.0f}')

In [ ]:
fig = plot_publication_trends(df, freq='W', save_path=str(OUTPUT_DIR / 'publication_trends_weekly.png'))
plt.show()

In [ ]:
fig = plot_publication_trends(df, freq='ME', save_path=str(OUTPUT_DIR / 'publication_trends_monthly.png'))
plt.show()

### Top 10 Highest-Volume Days

In [ ]:
top_days = top_publishing_days(df, top_n=10)
print('Days with the most published articles:')
print(top_days.to_string(index=False))

## 4. Text Analysis — Keywords & Topic Modeling

In [ ]:
from src.eda import top_count_terms, top_tfidf_terms, plot_top_terms, lda_topics

unigrams = top_count_terms(df, top_n=30, ngram_range=(1, 1))
bigrams  = top_count_terms(df, top_n=30, ngram_range=(2, 2))

print("Top 20 single keywords:")
print(unigrams.head(20).to_string())

In [ ]:
fig = plot_top_terms(unigrams.head(20), title="Top 20 Keywords in Headlines (Count)",
                     save_path=str(OUTPUT_DIR / "top_keywords.png"))
plt.show()

In [ ]:
fig = plot_top_terms(bigrams.head(20), title="Top 20 Bigrams in Headlines (Count)",
                     save_path=str(OUTPUT_DIR / "top_bigrams.png"))
plt.show()

### LDA Topic Modeling

In [ ]:
topics = lda_topics(df, n_topics=8, top_words=10)
for t in topics:
    print(f"Topic {t['topic']}: {chr(44).join(t['words'])}")

## 5. Time Series — Publication Volume & Publishing Hours

In [ ]:
from src.eda import plot_publishing_hours, publishing_hour_distribution

fig = plot_publication_trends(df, freq="W",
                              save_path=str(OUTPUT_DIR / "publication_trends_weekly.png"))
plt.show()

In [ ]:
hour_counts = publishing_hour_distribution(df)
peak_hour = hour_counts.idxmax()
print(f"Peak publishing hour (UTC): {peak_hour}:00  ({hour_counts[peak_hour]:,} articles)")
print(f"Quietest hour (UTC): {hour_counts.idxmin()}:00  ({hour_counts.min():,} articles)")

fig = plot_publishing_hours(df, save_path=str(OUTPUT_DIR / "publishing_hours.png"))
plt.show()

## 6. Publisher Analysis

In [ ]:
from src.eda import extract_publisher_domains, plot_publisher_domains, publisher_coverage_profile

print(f"Total unique publishers: {df['publisher'].nunique()}")
email_pubs = df[df["publisher"].str.contains("@", na=False)]
print(f"Email-format publishers: {email_pubs['publisher'].nunique()}")
print(f"Name-format publishers:  {df['publisher'].nunique() - email_pubs['publisher'].nunique()}")

In [ ]:
fig = plot_articles_per_publisher(df, top_n=20,
                                  save_path=str(OUTPUT_DIR / "articles_per_publisher.png"))
plt.show()

In [ ]:
domains = extract_publisher_domains(df)
print(f"Unique email domains: {len(domains)}")
print("\nTop 15 domains:")
print(domains.head(15).to_string())

In [ ]:
fig = plot_publisher_domains(df, top_n=15,
                            save_path=str(OUTPUT_DIR / "publisher_domains.png"))
plt.show()

In [ ]:
profile = publisher_coverage_profile(df, top_n=10)
print("Publisher coverage profiles:")
print(profile.to_string(index=False))